In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window


In [6]:
filepath = "D:\BDA 20\ABD-BDA-LabWork\datasets\sf-fire-calls.csv"

def create_session():
    spark = SparkSession.builder.appName('Fire Example').getOrCreate()
    return spark


In [7]:
#loads dataset to dataframes
#call type = key here

#header = True    ---- it means there is header so, ignore first row and consider it as column name.
# inferSchema = True         --- we asked to pyspark, to figured out schema by himself.

def create_df(spark, filepath):
    df = spark.read.csv(filepath, header = True, inferSchema = True)
    df1 = df.select('CallType', 'CallDate', 'City', 'Zipcode', 'Neighborhood', 'Delay')      #we filter column that we needed
    return df1


In [8]:
def clean_dataSet(df):
    df1 = df.withColumn('Date', to_date(col('CallDate'), 'MM/dd/yyyy')).drop('CallDate')
    df2 = df1.withColumn('Year',year(col('Date')))\
    .withColumn('Month',month(col('Date')))\
    .withColumn('Week',weekofyear(col('Date')))

    return df2


In [9]:
spark = create_session()
df = create_df(spark, filepath)
df = clean_dataSet(df)
df.printSchema()


root
 |-- CallType: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Delay: double (nullable = true)
 |-- Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Week: integer (nullable = true)



In [10]:
#Create a user defined function - maps Month to Season

def mapSeason(data):
    if 2 < data < 6:
        return 'Spring'
    elif 5 < data < 9:
        return 'Summer'
    elif 8 < data < 12:
        return 'Autumn'
    else:
        return 'Winter'

seasonUDF = udf(mapSeason, StringType())
clean = df.withColumn('Season', seasonUDF(col('Month')))
clean.show()


+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|        CallType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|Season|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|Winter|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|Winter|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94102

In [11]:
# selected year used by the questions that ask for "the selected year"
selected_year = 2018


## Q1: Yearly count of fire calls

In [12]:
def yearly_count(data):
    return data.groupBy('Year').count().orderBy('Year')

yearly_count(clean).show(30)


+----+-----+
|Year|count|
+----+-----+
|2000| 5459|
|2001| 7713|
|2002| 8090|
|2003| 8499|
|2004| 8283|
|2005| 8282|
|2006| 8174|
|2007| 8255|
|2008| 8869|
|2009| 8789|
|2010| 9341|
|2011| 9735|
|2012| 9674|
|2013|10020|
|2014|10775|
|2015|11458|
|2016|11609|
|2017|12135|
|2018|10136|
+----+-----+



## Q2: Different types of fire calls in 2018

In [13]:
def calltypes_in_year(data, yr):
    return data.filter(col('Year') == yr).select('CallType').distinct()

calltypes_in_year(clean, 2018).show(30, truncate=False)


+-------------------------------+
|CallType                       |
+-------------------------------+
|Elevator / Escalator Rescue    |
|Alarms                         |
|Odor (Strange / Unknown)       |
|Citizen Assist / Service Call  |
|HazMat                         |
|Vehicle Fire                   |
|Other                          |
|Outside Fire                   |
|Traffic Collision              |
|Assist Police                  |
|Gas Leak (Natural and LP Gases)|
|Water Rescue                   |
|Electrical Hazard              |
|Structure Fire                 |
|Medical Incident               |
|Fuel Spill                     |
|Smoke Investigation (Outside)  |
|Train / Rail Incident          |
|Explosion                      |
|Suspicious Package             |
+-------------------------------+



## Q3: Which week in 2018 had the most fire calls

In [14]:
def busiest_week(data, yr):
    return data.filter(col('Year') == yr) \
        .groupBy('Week').count().orderBy(desc('count'))

week_counts_2018 = busiest_week(clean, 2018)
week_counts_2018.show(5)

top_week = week_counts_2018.first()
print(f"Week {top_week['Week']} of 2018 had the most fire calls: {top_week['count']}")


+----+-----+
|Week|count|
+----+-----+
|  22|  259|
|  40|  255|
|  43|  250|
|  25|  249|
|   1|  246|
+----+-----+
only showing top 5 rows

Week 22 of 2018 had the most fire calls: 259


## Q4: Monthly count of fire calls based on year

In [15]:
def monthly_count(data):
    return data.groupBy('Year', 'Month').count().orderBy('Year', 'Month')

monthly_count(clean).show(50)


+----+-----+-----+
|Year|Month|count|
+----+-----+-----+
|2000|    4|  335|
|2000|    5|  680|
|2000|    6|  585|
|2000|    7|  668|
|2000|    8|  678|
|2000|    9|  655|
|2000|   10|  620|
|2000|   11|  595|
|2000|   12|  643|
|2001|    1|  622|
|2001|    2|  613|
|2001|    3|  692|
|2001|    4|  636|
|2001|    5|  682|
|2001|    6|  672|
|2001|    7|  646|
|2001|    8|  660|
|2001|    9|  577|
|2001|   10|  673|
|2001|   11|  619|
|2001|   12|  621|
|2002|    1|  726|
|2002|    2|  681|
|2002|    3|  665|
|2002|    4|  630|
|2002|    5|  621|
|2002|    6|  660|
|2002|    7|  669|
|2002|    8|  677|
|2002|    9|  671|
|2002|   10|  686|
|2002|   11|  683|
|2002|   12|  721|
|2003|    1|  721|
|2003|    2|  631|
|2003|    3|  729|
|2003|    4|  656|
|2003|    5|  723|
|2003|    6|  737|
|2003|    7|  694|
|2003|    8|  753|
|2003|    9|  717|
|2003|   10|  716|
|2003|   11|  683|
|2003|   12|  739|
|2004|    1|  686|
|2004|    2|  685|
|2004|    3|  707|
|2004|    4|  735|
|2004|    5|

## Q5: Monthly report of fire call types for the selected year

In [16]:
def monthly_calltype_report(data, yr):
    return data.filter(col('Year') == yr) \
        .groupBy('Month', 'CallType').count() \
        .orderBy('Month', desc('count'))

monthly_calltype_report(clean, selected_year).show(100, truncate=False)


+-----+-------------------------------+-----+
|Month|CallType                       |count|
+-----+-------------------------------+-----+
|1    |Medical Incident               |692  |
|1    |Alarms                         |122  |
|1    |Structure Fire                 |91   |
|1    |Traffic Collision              |42   |
|1    |Citizen Assist / Service Call  |15   |
|1    |Outside Fire                   |14   |
|1    |Gas Leak (Natural and LP Gases)|5    |
|1    |Water Rescue                   |4    |
|1    |Vehicle Fire                   |4    |
|1    |Other                          |3    |
|1    |Electrical Hazard              |3    |
|1    |Smoke Investigation (Outside)  |3    |
|1    |Elevator / Escalator Rescue    |3    |
|1    |Train / Rail Incident          |2    |
|1    |Odor (Strange / Unknown)       |2    |
|1    |HazMat                         |1    |
|1    |Fuel Spill                     |1    |
|2    |Medical Incident               |635  |
|2    |Alarms                     

## Q6: Top 5 fire call types for every season of the selected year

In [17]:
def top5_calltypes_by_season(data, yr):
    season_counts = data.filter(col('Year') == yr) \
        .groupBy('Season', 'CallType').count()

    w = Window.partitionBy('Season').orderBy(desc('count'))
    return season_counts.withColumn('rank', row_number().over(w)) \
        .filter(col('rank') <= 5) \
        .orderBy('Season', 'rank')

top5_calltypes_by_season(clean, selected_year).show(40, truncate=False)


+------+-----------------+-----+----+
|Season|CallType         |count|rank|
+------+-----------------+-----+----+
|Autumn|Medical Incident |1514 |1   |
|Autumn|Alarms           |251  |2   |
|Autumn|Structure Fire   |201  |3   |
|Autumn|Traffic Collision|100  |4   |
|Autumn|Outside Fire     |39   |5   |
|Spring|Medical Incident |2110 |1   |
|Spring|Alarms           |333  |2   |
|Spring|Structure Fire   |261  |3   |
|Spring|Traffic Collision|133  |4   |
|Spring|Other            |36   |5   |
|Summer|Medical Incident |2053 |1   |
|Summer|Alarms           |336  |2   |
|Summer|Structure Fire   |262  |3   |
|Summer|Traffic Collision|121  |4   |
|Summer|Outside Fire     |61   |5   |
|Winter|Medical Incident |1327 |1   |
|Winter|Alarms           |224  |2   |
|Winter|Structure Fire   |182  |3   |
|Winter|Traffic Collision|79   |4   |
|Winter|Outside Fire     |28   |5   |
+------+-----------------+-----+----+



## Q7: Are fire type calls seasonal?\n\nCompare each call type's share of calls across the four seasons (all years). If a call type's percentage varies noticeably across seasons it is seasonal; if it stays roughly flat, it is not.

In [18]:
def calltype_season_share(data):
    season_totals = data.groupBy('Season').count().withColumnRenamed('count', 'season_total')
    calltype_season = data.groupBy('Season', 'CallType').count()

    return calltype_season.join(season_totals, on='Season') \
        .withColumn('pct_of_season', round(col('count') / col('season_total') * 100, 2)) \
        .orderBy('CallType', 'Season')

calltype_season_share(clean).show(50, truncate=False)


+------+--------------------------------------------+-----+------------+-------------+
|Season|CallType                                    |count|season_total|pct_of_season|
+------+--------------------------------------------+-----+------------+-------------+
|Autumn|Administrative                              |1    |44264       |0.0          |
|Spring|Administrative                              |1    |43821       |0.0          |
|Winter|Administrative                              |1    |42770       |0.0          |
|Autumn|Aircraft Emergency                          |10   |44264       |0.02         |
|Spring|Aircraft Emergency                          |6    |43821       |0.01         |
|Summer|Aircraft Emergency                          |10   |44441       |0.02         |
|Winter|Aircraft Emergency                          |10   |42770       |0.02         |
|Autumn|Alarms                                      |4980 |44264       |11.25        |
|Spring|Alarms                             

## Q8: Which months in 2018 saw the highest number of fire calls

In [19]:
def monthly_count_in_year(data, yr):
    return data.filter(col('Year') == yr) \
        .groupBy('Month').count().orderBy(desc('count'))

months_2018 = monthly_count_in_year(clean, 2018)
months_2018.show(12)

top_month_2018 = months_2018.first()
print(f"Highest month in 2018: Month {top_month_2018['Month']} ({top_month_2018['count']} calls)")


+-----+-----+
|Month|count|
+-----+-----+
|   10| 1068|
|    5| 1047|
|    3| 1029|
|    8| 1021|
|    1| 1007|
|    6|  974|
|    7|  974|
|    9|  951|
|    4|  947|
|    2|  919|
|   11|  199|
+-----+-----+

Highest month in 2018: Month 10 (1068 calls)


## Q9: Major (most frequent) call type in each year

In [20]:
def major_calltype_per_year(data):
    year_calltype_counts = data.groupBy('Year', 'CallType').count()
    w = Window.partitionBy('Year').orderBy(desc('count'))
    return year_calltype_counts.withColumn('rank', row_number().over(w)) \
        .filter(col('rank') == 1) \
        .drop('rank') \
        .orderBy('Year')

major_calltype_per_year(clean).show(30, truncate=False)


+----+----------------+-----+
|Year|CallType        |count|
+----+----------------+-----+
|2000|Medical Incident|3408 |
|2001|Medical Incident|4653 |
|2002|Medical Incident|5046 |
|2003|Medical Incident|5056 |
|2004|Medical Incident|5137 |
|2005|Medical Incident|5084 |
|2006|Medical Incident|5027 |
|2007|Medical Incident|5114 |
|2008|Medical Incident|5692 |
|2009|Medical Incident|5671 |
|2010|Medical Incident|6186 |
|2011|Medical Incident|6413 |
|2012|Medical Incident|6296 |
|2013|Medical Incident|6690 |
|2014|Medical Incident|7176 |
|2015|Medical Incident|7812 |
|2016|Medical Incident|7999 |
|2017|Medical Incident|8330 |
|2018|Medical Incident|7004 |
+----+----------------+-----+



## Q10: Average delay in response for each call type

In [21]:
def avg_delay_by_calltype(data):
    return data.groupBy('CallType') \
        .agg(round(avg('Delay'), 2).alias('avg_delay')) \
        .orderBy(desc('avg_delay'))

delay_by_type = avg_delay_by_calltype(clean)
delay_by_type.show(30, truncate=False)


+--------------------------------------------+---------+
|CallType                                    |avg_delay|
+--------------------------------------------+---------+
|Mutual Aid / Assist Outside Agency          |38.42    |
|Assist Police                               |26.98    |
|Train / Rail Incident                       |16.45    |
|Administrative                              |12.26    |
|HazMat                                      |7.53     |
|Marine Fire                                 |6.93     |
|Confined Space / Structure Collapse         |6.92     |
|Watercraft in Distress                      |6.89     |
|Suspicious Package                          |6.58     |
|High Angle Rescue                           |6.05     |
|Other                                       |5.51     |
|Water Rescue                                |5.51     |
|Fuel Spill                                  |5.49     |
|Citizen Assist / Service Call               |5.47     |
|Electrical Hazard             

## Q11: Call type with the maximum average delay time

In [22]:
max_delay_calltype = delay_by_type.first()
print(f"Call type with maximum average delay: {max_delay_calltype['CallType']} "
      f"({max_delay_calltype['avg_delay']} minutes)")


Call type with maximum average delay: Mutual Aid / Assist Outside Agency (38.42 minutes)


## Q12: Neighborhood with the most fire calls in 2018

In [23]:
def calls_by_neighborhood(data, yr):
    return data.filter(col('Year') == yr) \
        .groupBy('Neighborhood').count().orderBy(desc('count'))

neighborhood_counts_2018 = calls_by_neighborhood(clean, 2018)
neighborhood_counts_2018.show(10, truncate=False)

top_neighborhood_2018 = neighborhood_counts_2018.first()
print(f"Neighborhood with most fire calls in 2018: {top_neighborhood_2018['Neighborhood']} "
      f"({top_neighborhood_2018['count']} calls)")


+------------------------------+-----+
|Neighborhood                  |count|
+------------------------------+-----+
|Tenderloin                    |1393 |
|South of Market               |1053 |
|Mission                       |913  |
|Financial District/South Beach|772  |
|Bayview Hunters Point         |522  |
|Western Addition              |352  |
|Sunset/Parkside               |346  |
|Nob Hill                      |295  |
|Hayes Valley                  |291  |
|Outer Richmond                |262  |
+------------------------------+-----+
only showing top 10 rows

Neighborhood with most fire calls in 2018: Tenderloin (1393 calls)


## Q13: Neighborhoods with the worst (highest) response times in 2018

In [24]:
def delay_by_neighborhood(data, yr):
    return data.filter(col('Year') == yr) \
        .groupBy('Neighborhood') \
        .agg(round(avg('Delay'), 2).alias('avg_delay'), count('*').alias('num_calls')) \
        .orderBy(desc('avg_delay'))

delay_by_neighborhood(clean, 2018).show(10, truncate=False)


+------------------------------+---------+---------+
|Neighborhood                  |avg_delay|num_calls|
+------------------------------+---------+---------+
|Chinatown                     |6.19     |191      |
|Presidio                      |5.83     |69       |
|Treasure Island               |5.45     |72       |
|McLaren Park                  |4.74     |14       |
|Bayview Hunters Point         |4.62     |522      |
|Presidio Heights              |4.59     |71       |
|Inner Sunset                  |4.44     |154      |
|Inner Richmond                |4.36     |129      |
|Financial District/South Beach|4.34     |772      |
|Haight Ashbury                |4.27     |140      |
+------------------------------+---------+---------+
only showing top 10 rows



## Q14: Trend of average response delay over years, per call type\n\nCorrelation between Year and avg_delay per call type: positive -> delay increases over years, negative -> decreases, near 0 -> no relation.

In [25]:
def delay_trend_by_calltype(data):
    yearly_delay = data.groupBy('Year', 'CallType') \
        .agg(round(avg('Delay'), 2).alias('avg_delay'))

    return yearly_delay.groupBy('CallType') \
        .agg(round(corr('Year', 'avg_delay'), 3).alias('year_delay_correlation')) \
        .orderBy(desc('year_delay_correlation'))

trend = delay_trend_by_calltype(clean)
trend.show(30, truncate=False)

# trend specifically for the call type with the overall maximum average delay (Q11)
trend.filter(col('CallType') == max_delay_calltype['CallType']).show(truncate=False)


+--------------------------------------------+----------------------+
|CallType                                    |year_delay_correlation|
+--------------------------------------------+----------------------+
|Mutual Aid / Assist Outside Agency          |0.919                 |
|Water Rescue                                |0.768                 |
|Alarms                                      |0.601                 |
|Suspicious Package                          |0.559                 |
|Other                                       |0.555                 |
|Odor (Strange / Unknown)                    |0.511                 |
|Traffic Collision                           |0.493                 |
|Smoke Investigation (Outside)               |0.463                 |
|Watercraft in Distress                      |0.391                 |
|Explosion                                   |0.388                 |
|Train / Rail Incident                       |0.376                 |
|Citizen Assist / Se

## Q15: For each year, which city has the most distinct call types

In [26]:
def top_city_by_calltypes(data):
    year_city_calltypes = data.groupBy('Year', 'City') \
        .agg(count_distinct('CallType').alias('distinct_calltypes'))

    w = Window.partitionBy('Year').orderBy(desc('distinct_calltypes'))
    return year_city_calltypes.withColumn('rank', row_number().over(w)) \
        .filter(col('rank') == 1) \
        .drop('rank') \
        .orderBy('Year')

top_city_by_calltypes(clean).show(30, truncate=False)


+----+-------------+------------------+
|Year|City         |distinct_calltypes|
+----+-------------+------------------+
|2000|SF           |18                |
|2001|SF           |20                |
|2002|SF           |20                |
|2003|SF           |24                |
|2004|SF           |23                |
|2005|SF           |26                |
|2006|SF           |24                |
|2007|SF           |26                |
|2008|SF           |23                |
|2009|SF           |22                |
|2010|SF           |23                |
|2011|SF           |25                |
|2012|SF           |25                |
|2013|SF           |24                |
|2014|San Francisco|21                |
|2015|San Francisco|24                |
|2016|San Francisco|24                |
|2017|San Francisco|26                |
|2018|San Francisco|20                |
+----+-------------+------------------+



## Q16: For every year, call-type count for the top 5 (busiest) cities

In [27]:
def top5_cities_calltype_counts(data):
    year_city_calltypes = data.groupBy('Year', 'City') \
        .agg(count_distinct('CallType').alias('distinct_calltypes'))
    year_city_total_calls = data.groupBy('Year', 'City').agg(count('*').alias('total_calls'))

    w = Window.partitionBy('Year').orderBy(desc('total_calls'))
    top5_cities = year_city_total_calls.withColumn('rank', row_number().over(w)) \
        .filter(col('rank') <= 5) \
        .drop('rank')

    return top5_cities.join(year_city_calltypes, on=['Year', 'City']) \
        .orderBy('Year', desc('total_calls'))

top5_cities_calltype_counts(clean).show(100, truncate=False)


+----+-------------+-----------+------------------+
|Year|City         |total_calls|distinct_calltypes|
+----+-------------+-----------+------------------+
|2000|SF           |5435       |18                |
|2000|TI           |8          |6                 |
|2000|YB           |5          |1                 |
|2000|HP           |3          |2                 |
|2000|SFO          |3          |1                 |
|2001|SF           |7656       |20                |
|2001|TI           |37         |6                 |
|2001|YB           |6          |2                 |
|2001|SFO          |4          |1                 |
|2001|DC           |3          |2                 |
|2002|SF           |8044       |20                |
|2002|TI           |18         |5                 |
|2002|PR           |7          |3                 |
|2002|HP           |6          |3                 |
|2002|SFO          |5          |1                 |
|2003|SF           |8441       |24                |
|2003|TI    